# IMT-342 — Hito 2: integración reproducible del ABB IRB 120

**Notebook integrador (versión de autocontrol, 19 de octubre de 2026).** Se apoya en los módulos del directorio `src/` del Hito 2. Su propósito es conectar la cinemática inversa de Pieper, la verificación FK, el Jacobiano, las singularidades, la trayectoria LSPB y la dinámica pedagógica reducida 1R.

**Material oficial:** `S10_material_oficial.pdf`, guía del Hito 2 (pp. 31–32), rúbrica (pp. 34–36); esquema de Notebook `S09_material_oficial.pdf`, p. 23. Las convenciones DH y el TCP provienen del modelo validado en el Hito 1, según `docs/arquitectura_y_convenciones.md`.

**Distinción de procedencia:** las poses de autocontrol, los tiempos de planificación y las tolerancias numéricas de este cuaderno son elecciones del equipo para validar el software. **No son poses asignadas por el docente.** El caso dinámico 1R sí utiliza los valores didácticos de W10-S01 (S10, pp. 16–17), que no corresponden a parámetros reales del fabricante ABB.

**Ejecución:** abrir este archivo desde `hito2/notebooks/` (o desde la raíz `hito2/`) y ejecutar **Kernel → Restart Kernel and Run All Cells**. No requiere ROS, internet ni datos externos.

## 1. Planteamiento y modelo del robot

Manipulador ABB IRB 120 de seis juntas revolutas. Se usa **DH estándar**, longitud en metros, ángulos en radianes, matrices \({}^{0}T_6\) de la brida, y velocidades espaciales \([{}^0\mathbf v;{}^0\boldsymbol\omega]\) expresadas en la base \(\{0\}\).

| Junta | \(\theta\) | \(d\) [m] | \(a\) [m] | \(\alpha\) |
|---|---|---:|---:|---|
| J1 | \(q_1\) | 0.290 | 0 | \(-\pi/2\) |
| J2 | \(q_2-\pi/2\) | 0 | 0.270 | 0 |
| J3 | \(q_3\) | 0 | 0.070 | \(-\pi/2\) |
| J4 | \(q_4\) | 0.302 | 0 | \(+\pi/2\) |
| J5 | \(q_5\) | 0 | 0 | \(-\pi/2\) |
| J6 | \(q_6\) | 0.072 | 0 | 0 |

El TCP académico posee un desplazamiento de **+0.120 m sobre \(z_6\)** respecto de la brida. En este primer experimento todas las poses objetivo se refieren a la **brida** (`target_frame="flange"`) para evitar mezclar marcos.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "pieper.py").is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / "src" / "pieper.py").is_file():
    raise RuntimeError("Ejecutar desde hito2/ o hito2/notebooks/; no se encontró src/pieper.py")
sys.path.insert(0, str(RAIZ)) if str(RAIZ) not in sys.path else None

from src.irb120_fk import forward_kinematics
from src.pieper import solve_pieper
from src.seleccion_pieper import filtrar_y_seleccionar_pieper
from src.lspb import planificar_lspb, VELOCIDADES_ABB_RAD_S
from src.jacobiano import jacobiano_geometrico, velocidad_cartesiana
from src.singularidades import analizar_singularidad, experimento_singularidad_muneca
from src.dinamica_1r import ParametrosDinamica1R, evaluar_dinamica_1r

np.set_printoptions(precision=6, suppress=True)
print("Raíz del Hito 2:", RAIZ)
print("NumPy:", np.__version__)
print("Intérprete:", sys.executable)

In [ ]:
# Parámetros de autocontrol del equipo, NO consignas oficiales.
q_inicio = np.zeros(6)
q_referencia = np.array([0.30, -0.20, 0.10, 0.20, -0.40, 0.50])  # rad
T_objetivo = forward_kinematics(q_referencia)
print("Configuración de autocontrol [rad]:", q_referencia)
print("Pose objetivo de brida 0T6 (m):\n", T_objetivo)
assert T_objetivo.shape == (4, 4)
assert np.allclose(T_objetivo[3], [0, 0, 0, 1], atol=1e-12)

## 2. Cinemática inversa analítica: Pieper

Separamos el **enumerador matemático de ramas** de la **selección sujeta a límites articulares nominales ABB**. La solución de Pieper utiliza la estructura geométrica de muñeca esférica. La búsqueda numérica genérica de IK no interviene en `solve_pieper`.

Una rama regular puede distinguir elecciones de brazo y de muñeca. Las soluciones singulares retornadas por el solver son representantes de familias continuas, no una enumeración exhaustiva de infinitas soluciones.

In [ ]:
resultado_ik = solve_pieper(T_objetivo, target_frame="flange", q_seed=q_inicio)
resultado_sel = filtrar_y_seleccionar_pieper(resultado_ik, q_inicio)
assert resultado_ik.solutions, "Pieper no devolvió ramas para el autocontrol"
assert resultado_sel.seleccionada is not None, "No hay ninguna solución dentro de los límites ABB"

print(f"Ramas matemáticas verificadas por Pieper: {len(resultado_ik.solutions)}")
print(f"Representaciones admisibles tras filtrado ABB: {len(resultado_sel.admisibles)}")
print(f"Ramas descartadas: {len(resultado_sel.descartadas)}")
print("\nRama | brazo | muñeca | error posición [m] | error orientación [rad] | singular")
for i, sol in enumerate(resultado_ik.solutions, 1):
    print(f"{i:4d} | {sol.arm_branch:>6} | {sol.wrist_branch:>6} | "
          f"{sol.position_error_m:.3e} | {sol.orientation_error_rad:.3e} | {sol.singular}")

q_final = resultado_sel.seleccionada.q.copy()
print("\nConfiguración seleccionada [rad]:", q_final)
print("Rama elegida:", resultado_sel.seleccionada.rama_brazo,
      resultado_sel.seleccionada.rama_muneca)
print("Costo articular ponderado:", resultado_sel.seleccionada.costo)
if resultado_sel.diagnosticos:
    print("Diagnósticos:", resultado_sel.diagnosticos)

## 3. Verificación de las soluciones mediante FK

Para cada \(\mathbf q_k\) aceptada por Pieper se reconstruye \({}^{0}T_6(\mathbf q_k)\) mediante FK, comprobando **posición y orientación**. Para una matriz de rotación \(R\), el error angular puede medirse mediante \(\theta=\cos^{-1}((\operatorname{tr}(R_d^\top R)-1)/2)\), recortando el argumento al intervalo \([-1,1]\). Esta comprobación adicional del Notebook no reemplaza los controles internos del solver.

In [ ]:
def errores_fk(T_observada, T_deseada):
    ep = float(np.linalg.norm(T_observada[:3, 3] - T_deseada[:3, 3]))
    c = (np.trace(T_deseada[:3, :3].T @ T_observada[:3, :3]) - 1.0) / 2.0
    er = float(np.arccos(np.clip(c, -1.0, 1.0)))
    return ep, er

errores = np.array([errores_fk(forward_kinematics(sol.q), T_objetivo)
                    for sol in resultado_ik.solutions])
print("Error máximo de posición de las ramas [m]:", np.max(errores[:, 0]))
print("Error máximo angular de las ramas [rad]:", np.max(errores[:, 1]))
assert np.all(errores[:, 0] < 1e-7)
assert np.all(errores[:, 1] < 1e-6)

error_pos_elegida, error_ang_elegida = errores_fk(forward_kinematics(q_final), T_objetivo)
print("Rama elegida: error posición [m] =", error_pos_elegida)
print("Rama elegida: error orientación [rad] =", error_ang_elegida)
assert error_pos_elegida < 1e-7 and error_ang_elegida < 1e-6

## 4. Jacobiano geométrico y comprobación manual

Para la junta revoluta \(i\), referida al marco base:

\[
{}^0J_{v,i}={}^0\mathbf z_{i-1}\times({}^0\mathbf p_e-{}^0\mathbf o_{i-1}),
\qquad {}^0J_{\omega,i}={}^0\mathbf z_{i-1}.
\]

El Jacobiano de la brida es \(6\times 6\) y ordena primero las velocidades lineales (m/s), después las angulares (rad/s). Para **J1**, el eje \({}^0\mathbf z_0=[0,0,1]^\top\) y el origen de su eje coincide con el origen base; calculamos esa columna manualmente, de manera independiente de `jacobiano_geometrico`.

In [ ]:
J = jacobiano_geometrico(q_final, target_frame="flange")
p_efector = forward_kinematics(q_final)[:3, 3]
z0 = np.array([0.0, 0.0, 1.0])
columna_j1_manual = np.r_[np.cross(z0, p_efector), z0]
print("Columna J1 calculada manualmente:", columna_j1_manual)
print("Columna J1 implementada:          ", J[:, 0])
assert J.shape == (6, 6)
np.testing.assert_allclose(J[:, 0], columna_j1_manual, atol=1e-11)
print("Comprobación manual J1: SATISFACTORIA")

## 5. Singularidades: pérdida de rango y velocidades articulares

El experimento independiente aproxima \(q_5\to0\) con las otras cinco juntas fijas. En la singularidad de muñeca los ejes de J4 y J6 quedan alineados y **el rango del Jacobiano baja de 6 a 5**. Los valores singulares se diagnostican con una longitud de referencia \(L=1\,\mathrm m\) para normalizar magnitudes lineales y angulares; esta elección es del equipo y no altera la condición de singularidad exacta.

Se utiliza una **misma consigna cartesiana fija** para mostrar la amplificación articular al acercarse a la singularidad; dichas velocidades son matemáticas y **no representan órdenes seguras para el robot físico**.

In [ ]:
exp_sing = experimento_singularidad_muneca()
print(" q5 [rad]  rango    sigma_min(normalizada)       norma qdot [rad/s]")
for q5, rango, sigma, norma in zip(exp_sing.angulos_q5_rad,
                                   exp_sing.rangos,
                                   exp_sing.sigma_min,
                                   exp_sing.normas_qdot):
    print(f"{q5:8.5f}    {rango:d}       {sigma:12.5e}           {norma:12.5g}")
assert exp_sing.rangos[-1] == 5
assert np.all(exp_sing.rangos[:-1] == 6)
assert np.isnan(exp_sing.normas_qdot[-1])
print("Dirección cartesiana normalizada perdida:",
      exp_sing.direccion_perdida_normalizada)
print("Modo nulo articular:", exp_sing.modo_nulo_articular)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.7))
axes[0].plot(exp_sing.angulos_q5_rad, exp_sing.rangos, "o-")
axes[0].set(ylabel="Rango", xlabel="q5 [rad]", title="Pérdida de rango")
axes[1].semilogy(exp_sing.angulos_q5_rad, exp_sing.sigma_min, "o-")
axes[1].set(ylabel="Valor singular mínimo normalizado", xlabel="q5 [rad]",
            title="Aproximación a singularidad")
mask = np.isfinite(exp_sing.normas_qdot)
axes[2].semilogy(exp_sing.angulos_q5_rad[mask], exp_sing.normas_qdot[mask], "o-")
axes[2].set(ylabel="Norma qdot [rad/s]", xlabel="q5 [rad]",
            title="Amplificación articular")
for ax in axes: ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Generación de trayectoria LSPB de seis juntas

Se planifica movimiento de reposo a reposo entre \(\mathbf q_{inicio}\) y la configuración elegida por Pieper. La aceleración de **1 rad/s² por junta** y el tiempo \(t_f=8\,\mathrm s\) son parámetros de **autocontrol académico**, no límites de aceleración certificados de ABB. `planificar_lspb` comprueba límites nominales de posición y velocidad para 6R.

Se muestran, en tres ejes alineados, \(q(t)\), \(\dot q(t)\) y \(\ddot q(t)\) para las seis juntas. En los cambios de fase, la aceleración ideal es discontinua.

In [ ]:
perfil_6r = planificar_lspb(q_inicio, q_final, 8.0, np.ones(6))
t_6r = np.linspace(0.0, perfil_6r.tf, 401)
tray = perfil_6r.evaluar(t_6r)
assert tray.q.shape == (401, 6)
np.testing.assert_allclose(tray.q[0], q_inicio, atol=1e-10)
np.testing.assert_allclose(tray.q[-1], q_final, atol=1e-10)
np.testing.assert_allclose(tray.qdot[[0, -1]], 0, atol=1e-10)
assert np.all(np.abs(tray.qdot) <= VELOCIDADES_ABB_RAD_S + 1e-10)
print("Tiempo total [s]:", perfil_6r.tf)
print("Tiempos de mezcla por junta [s]:", perfil_6r.tc)
print("Velocidades de crucero [rad/s]:", perfil_6r.v_crucero)
print("FK(q(tf)) coincide con pose objetivo:", errores_fk(forward_kinematics(tray.q[-1]), T_objetivo))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for j in range(6):
    etiqueta = f"J{j+1}"
    axes[0].plot(t_6r, tray.q[:, j], label=etiqueta)
    axes[1].plot(t_6r, tray.qdot[:, j], label=etiqueta)
    axes[2].plot(t_6r, tray.qddot[:, j], label=etiqueta)
for ax, ylabel in zip(axes,
    ["Posición [rad]", "Velocidad [rad/s]", "Aceleración [rad/s²]"]):
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
axes[0].legend(ncol=6, loc="best")
axes[0].set_title("Trayectorias LSPB de seis articulaciones")
axes[-1].set_xlabel("Tiempo [s]")
fig.tight_layout()
plt.show()

## 7. Dinámica pedagógica reducida 1R

**Alcance y fuente:** según el ejercicio oficial W10-S01, S10 pp. 16–17, se utiliza
\[
\tau(t)=J_{1R}\ddot q(t)+mg\ell_c\sin q(t)
=\tau_{inercial}(t)+\tau_{gravitatorio}(t).
\]

En este modelo particular \(J_{1R}\) es constante y no se introduce un término independiente de Coriolis. Esto **no describe la dinámica acoplada de seis articulaciones** del ABB IRB 120. El ejemplo es una trayectoria **1R propia del docente**, independiente del movimiento 6R de las secciones previas.

Parámetros didácticos: \(q_i=0\), \(q_f=2\,\mathrm{rad}\), \(t_f=3\,\mathrm s\), \(a=1\,\mathrm{rad/s^2}\), \(J_{1R}=0.25\,\mathrm{kg\,m^2}\), \(m=1\,\mathrm{kg}\), \(\ell_c=0.5\,\mathrm m\), \(g=9.81\,\mathrm{m/s^2}\).

In [ ]:
perfil_1r = planificar_lspb([0], [2], 3.0, [1.0])
param_1r = ParametrosDinamica1R(inercia_kg_m2=0.25,
                                masa_kg=1.0,
                                longitud_cm_m=0.5,
                                gravedad_m_s2=9.81)
t_1r = np.linspace(0.0, 3.0, 601)
din = evaluar_dinamica_1r(perfil_1r, t_1r, param_1r)
control = evaluar_dinamica_1r(perfil_1r, 2.5, param_1r)
print(f"Verificación docente en t=2.5 s: q={control.q_rad:.6f} rad, "
      f"qddot={control.qddot_rad_s2:.6f} rad/s², "
      f"tau={control.tau_total_nm:.6f} N·m")
np.testing.assert_allclose(control.q_rad, 1.875, atol=1e-12)
np.testing.assert_allclose(control.qddot_rad_s2, -1.0, atol=1e-12)
np.testing.assert_allclose(control.tau_total_nm,
                           -0.25 + 4.905*np.sin(1.875), atol=1e-10)

crucero = evaluar_dinamica_1r(perfil_1r, 1.5, param_1r)
assert abs(crucero.tau_inercial_nm) < 1e-12
print(f"Crucero en t=1.5 s: tau_inercial={crucero.tau_inercial_nm:.6f}, "
      f"tau_gravitatorio={crucero.tau_gravitatorio_nm:.6f} N·m")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(t_1r, din.tau_inercial_nm, label="Inercial")
axes[1].plot(t_1r, din.tau_gravitatorio_nm, label="Gravitatorio")
axes[2].plot(t_1r, din.tau_total_nm, label="Total")
for ax, etiqueta in zip(axes,
                       ["Torque inercial [N·m]", "Torque gravitatorio [N·m]",
                        "Torque total [N·m]"]):
    ax.set_ylabel(etiqueta)
    ax.axvline(1.0, linestyle="--", alpha=0.5)
    ax.axvline(2.0, linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)
axes[0].set_title("Modelo pedagógico 1R — transiciones LSPB en t=1 y 2 s")
axes[2].set_xlabel("Tiempo [s]")
fig.tight_layout()
plt.show()

## 8. Gráficas consolidadas y discusión técnica

El Jacobiano transforma **velocidades articulares** en el twist de la brida:
\[
{}^0\mathcal V_6(t)={}^0J_6(\mathbf q(t))\dot{\mathbf q}(t).
\]

Esta sección evalúa el twist de la trayectoria 6R seleccionada, sin diferenciar numéricamente el Jacobiano. Se comprueba que, cuando \(\dot{\mathbf q}=0\) al inicio y al final, el twist del efector también es nulo. **La gráfica de amplificación de singularidad anterior es un experimento matemático separado:** sus velocidades no se utilizan para controlar el robot.

In [ ]:
twist = np.array([velocidad_cartesiana(q, qd, target_frame="flange")
                  for q, qd in zip(tray.q, tray.qdot)])
assert twist.shape == (401, 6)
np.testing.assert_allclose(twist[[0, -1]], 0.0, atol=1e-9)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t_6r, np.linalg.norm(twist[:, :3], axis=1))
axes[1].plot(t_6r, np.linalg.norm(twist[:, 3:], axis=1))
axes[0].set(ylabel="||v|| [m/s]", title="Velocidad de brida mediante Jacobiano")
axes[1].set(xlabel="Tiempo [s]", ylabel="||ω|| [rad/s]")
for ax in axes: ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("Pico muestreado de velocidad lineal [m/s]:",
      float(np.max(np.linalg.norm(twist[:, :3], axis=1))))
print("Pico muestreado de velocidad angular [rad/s]:",
      float(np.max(np.linalg.norm(twist[:, 3:], axis=1))))

## 9. Validación y conclusiones

### Comprobaciones integradas efectuadas por este cuaderno

1. La pose de autocontrol proviene de FK, no de una consigna no verificada.
2. Pieper enumera soluciones y el filtro ABB registra soluciones admisibles y descartadas.
3. Cada rama devuelta se vuelve a evaluar con FK y se controlan errores de posición y orientación.
4. La columna geométrica J1 se contrasta manualmente con un producto cruz.
5. En el experimento de muñeca el rango cae a cinco en \(q_5=0\), y se observa amplificación articular antes de la singularidad.
6. LSPB comienza y termina en reposo y respeta las velocidades nominales usadas por el planificador.
7. La dinámica pedagógica 1R reproduce el torque de referencia docente en \(t=2.5\,\mathrm s\).

**Limitaciones:** no se verifican colisiones, obstáculos, torques físicos del ABB, aceleraciones nominales certificadas ni seguimiento de control. Pieper retorna representantes de familias singulares. El comportamiento cerca de una singularidad exige tratamiento especial; la inversión directa del Jacobiano en rango deficiente es inapropiada. Los saltos del perfil LSPB son propios de la idealización.

**Pendiente de entrada oficial:** sustituir el caso de autocontrol por las matrices de pose que asigne el docente, conservando los mismos criterios de evaluación y documentando para cada objetivo su alcanzabilidad, ramas, errores de FK y elección. No atribuir las poses de este Notebook al docente.

**Pruebas de regresión externas al Notebook:** desde la raíz `hito2/`, ejecutar `python -m unittest discover -s validacion -p 'test_*.py' -v` en el entorno IMT-342; la última ejecución aportada en Fedora informó **71/71 pruebas satisfactorias**. La ejecución limpia de este Notebook debe verificarse también en Fedora antes de registrarlo como validado.